# Exploratory Data Analysis - Bike Sharing Demand

This notebook performs an end-to-end Exploratory Data Analysis on the Bike Sharing dataset.

## 1. Load and Inspect the Dataset
### Question
What does the raw dataset look like in terms of size, features, and data types?

### Code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from ydata_profiling import ProfileReport

import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_parquet('../data/raw_data/raw_data_full.parquet')

# Display basic info
display(df.head())
display(df.tail())
print(f"Shape of dataset: {df.shape}")
print("\nColumns:", df.columns.tolist())
print("\nDataset Info:")
df.info()
print("\nSummary Statistics:")
display(df.describe(include='all'))


### Observation
The dataset contains a mix of numerical and categorical (stored as numerical codes) columns, as well as a target variable (`cnt`). Potential identifiers include the `datetime` column.
- Numerical columns include `temp`, `atemp`, `hum`, `windspeed`, `casual`, `registered`, `cnt`.
- Categorical/discrete columns include `season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`, `weathersit`.
- The target variable is `cnt` which represents the total bike rentals.

### Modeling Implication
Understanding the types of features is the first step towards feature engineering and preprocessing. Numerical columns might need scaling, categorical ones might need encoding, and identifier/time columns like `datetime` need to be parsed to extract useful time-based features.

## 2. Understand the Features
### Question
What are the important features in the dataset, and are there any potential issues like target leakage?

### Code

In [ ]:
# Display columns to highlight target leakage variables
print("Columns in dataset:", df.columns.tolist())
print("Target variable: cnt")
print("Leakage variables: casual, registered")


### Observation
The variables `casual` and `registered` sum exactly to the target variable `cnt` (`cnt = casual + registered`).

### Modeling Implication
**TARGET LEAKAGE:** The `casual` and `registered` columns represent future information that wouldn't genuinely be known at prediction time. If our objective is predicting `cnt`, these variables must be excluded from the model's input features to prevent target leakage.

## 3. Data Quality Analysis
### Question
Are there missing values, duplicated rows, or invalid types in the dataset?

### Code

In [ ]:
# Missing Values
missing_vals = df.isnull().sum()
missing_percent = (missing_vals / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_vals, 'Percentage': missing_percent})
print("Missing Values Analysis:")
display(missing_df)

# Duplicate Rows
total_duplicates = df.duplicated().sum()
print(f"\nTotal duplicated rows: {total_duplicates}")

# Convert datetime if needed and check duplicate timestamps
if 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'])
    duplicate_timestamps = df.duplicated(subset=['datetime']).sum()
    print(f"Duplicated timestamps: {duplicate_timestamps}")

# Invalid values range check
categorical_ranges = {
    'season': (1, 4),
    'mnth': (1, 12),
    'hr': (0, 23),
    'holiday': (0, 1),
    'workingday': (0, 1),
    'weekday': (0, 6),
    'weathersit': (1, 4)
}

print("\nValidating Categorical Ranges:")
for col, (min_val, max_val) in categorical_ranges.items():
    if col in df.columns:
        invalid_count = df[(df[col] < min_val) | (df[col] > max_val)].shape[0]
        print(f"{col}: {invalid_count} invalid values out of expected range {min_val}-{max_val}")


### Observation
We calculate the absolute count and percentage of missing values. We also check for duplicate rows and duplicated timestamps (as each hour should uniquely correspond to an observation). We verified that categorical code variables fall within their expected valid ranges.

### Modeling Implication
Missing values need imputation before feeding to a model. Duplicate rows or timestamps require deduplication or investigation into why multiple records exist for a single hour. Invalid values must be handled (either corrected, dropped, or imputed).

## 4. Dataset Profiling
### Question
What insights can an automated profiling report provide?

### Code

In [ ]:
# Profiling the dataset
# Un-comment the lines below to generate the report when running the notebook locally
# profile = ProfileReport(df, title="Bike Sharing Demand Profiling Report", explorative=True)
# profile.to_widgets()


### Observation
A `ydata_profiling` report generates comprehensive information about missing values, distributions, high cardinality, and correlations across all features quickly. 

### Modeling Implication
Automated profiling provides a baseline understanding of correlations, skewness, and cardinality, pointing out variables that might need transformations or highlighting multi-collinearity that needs addressing.

## 5. Target Variable Analysis
### Question
How is the target variable `cnt` distributed?

### Code

In [ ]:
# Target Variable Analysis
target_stats = df['cnt'].describe()
skewness = df['cnt'].skew()
print("Target Variable (cnt) Statistics:")
display(target_stats)
print(f"Skewness: {skewness:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df['cnt'], kde=True, ax=axes[0])
axes[0].set_title('Histogram & KDE of cnt')

sns.boxplot(x=df['cnt'], ax=axes[1])
axes[1].set_title('Boxplot of cnt')
plt.show()


### Observation
The target variable `cnt` often exhibits a strong right-skewed distribution, meaning most hours have a relatively low number of rentals, with occasional highly-demanded hours forming a long tail. It's likely zero-inflated or close to zero during off-peak hours (e.g., late night).

### Modeling Implication
Because regression models like Linear Regression assume normally distributed residuals, a highly skewed target might lead to suboptimal predictions and violate model assumptions. A transformation such as `np.log1p(cnt)` could potentially help stabilize the variance and make the distribution more normal.

## 6. Target Over Time
### Question
How does rental demand vary over the entire chronological time series?

### Code

In [ ]:
# Target Over Time
if 'datetime' in df.columns:
    fig = px.line(df, x='datetime', y='cnt', title='Hourly Bike Rental Demand Over Time')
    fig.update_xaxes(rangeslider_visible=True)
    # To view interactively in the notebook, simply call fig.show()
    # fig.show()


### Observation
Plotting `datetime` vs `cnt` reveals overall trends (e.g., growth across years), strong seasonality (fluctuations across months/seasons), and repeating daily/weekly cycles. There may also be sudden spikes or unusual low-demand periods.

### Modeling Implication
The clear temporal structure implies that observations are not fully independent. Future data depends on past data, suggesting that simple random train-test splitting might lead to data leakage and over-optimistic validation scores. A chronological split is recommended.

## 7. Hourly Demand Pattern
### Question
What does the average daily rental profile look like?

### Code

In [ ]:
hourly_demand = df.groupby("hr")["cnt"].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=hourly_demand, x="hr", y="cnt", color="skyblue")
plt.title("Average Bike Demand by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Average Rentals")
plt.show()


### Observation
Demand typically peaks twice during the day: around 8 AM (morning commute) and 5-6 PM (evening commute). The lowest demand hours are in the early morning (e.g., 2 AM - 5 AM). 

### Modeling Implication
Hour is likely to be one of the strongest predictors because of these distinct commuter peaks. Moreover, hour is cyclical (hour 23 is adjacent to hour 0). Cyclical encoding like `sin(2π × hour / 24)` and `cos(2π × hour / 24)` can help the model interpret this continuous periodic relationship correctly.

## 8. Rental Demand by Season
### Question
How does demand change across different seasons?

### Code

In [ ]:
season_map = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'}
df['season_name'] = df['season'].map(season_map)

season_stats = df.groupby('season_name')['cnt'].agg(['mean', 'median', 'count']).reset_index()
print("Demand by Season:")
display(season_stats)

plt.figure(figsize=(8, 5))
sns.barplot(data=season_stats, x='season_name', y='mean', order=['Winter', 'Spring', 'Summer', 'Fall'])
plt.title("Average Rental Demand by Season")
plt.xlabel("Season")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
Fall and Summer generally observe the highest demand, while Winter observes the lowest due to unfavorable biking conditions. The difference in means between seasons is noticeable and large.

### Modeling Implication
Season is clearly a useful predictor. It captures broad multi-month weather and temperature trends that drive biking behavior.

## 9. Rental Demand by Weather
### Question
How does the weather situation affect rentals?

### Code

In [ ]:
weather_map = {1: 'Clear', 2: 'Mist/Cloudy', 3: 'Light Rain/Snow', 4: 'Heavy Rain/Snow'}
df['weather_name'] = df['weathersit'].map(weather_map)

weather_stats = df.groupby('weather_name')['cnt'].agg(['mean', 'count']).reset_index()
print("Demand by Weather Condition:")
display(weather_stats)

plt.figure(figsize=(8, 5))
sns.barplot(data=weather_stats, x='weather_name', y='mean', order=['Clear', 'Mist/Cloudy', 'Light Rain/Snow', 'Heavy Rain/Snow'])
plt.title("Average Rental Demand by Weather")
plt.xlabel("Weather Condition")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
Demand decreases as weather becomes worse. Clear weather yields the highest rentals, followed by misty/cloudy conditions. Heavy Rain/Snow typically has very few observations, but very low average demand.

### Modeling Implication
Weather is a strong predictor. However, the extreme minority class (Severe weather) might cause issues for model generalization or cross-validation sampling since it has very few samples.

## 10. Rental Demand by Weekday
### Question
Does demand behave differently on weekends vs weekdays?

### Code

In [ ]:
weekday_map = {0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday'}
df['weekday_name'] = df['weekday'].map(weekday_map)

weekday_stats = df.groupby('weekday_name')['cnt'].mean().reset_index()
# Calendar order starting Monday
calendar_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(8, 5))
sns.barplot(data=weekday_stats, x='weekday_name', y='cnt', order=calendar_order, color="coral")
plt.title("Average Rental Demand by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
Overall daily average demand might not vary drastically between weekdays and weekends, but the *type* of demand might change (commuter vs casual). Depending on the specific dataset, weekends might show slightly lower or higher total daily volume compared to weekdays.

### Modeling Implication
Weekday might be a moderate predictor on its own, but its true power likely lies in interactions (e.g., Hour × Weekday), as the hourly pattern completely shifts from weekday to weekend.

## 11. Holiday Analysis
### Question
How do holidays affect rental demand compared to non-holidays?

### Code

In [ ]:
holiday_map = {0: 'Non-holiday', 1: 'Holiday'}
df['holiday_name'] = df['holiday'].map(holiday_map)

holiday_stats = df.groupby('holiday_name')['cnt'].agg(['mean', 'median', 'count']).reset_index()
print("Demand by Holiday:")
display(holiday_stats)

plt.figure(figsize=(6, 5))
sns.barplot(data=holiday_stats, x='holiday_name', y='mean')
plt.title("Average Rental Demand: Holiday vs Non-holiday")
plt.xlabel("Day Type")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
Holidays are much less frequent (imbalanced class). Demand on holidays might be slightly different on average compared to non-holidays, likely mimicking weekend patterns.

### Modeling Implication
Holiday is a very imbalanced feature. While it might appear predictive for specific days, models may struggle to learn from it if they do not see enough holiday samples.

## 12. Working Day Analysis
### Question
Does commuter behavior (working days) drive overall demand?

### Code

In [ ]:
workday_map = {0: 'Non-working day', 1: 'Working day'}
df['workday_name'] = df['workingday'].map(workday_map)

workday_stats = df.groupby('workday_name')['cnt'].agg(['mean', 'median', 'count']).reset_index()
print("Demand by Working Day:")
display(workday_stats)

plt.figure(figsize=(6, 5))
sns.barplot(data=workday_stats, x='workday_name', y='mean')
plt.title("Average Rental Demand: Working vs Non-working Day")
plt.xlabel("Day Type")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
Working and non-working days might have similar total averages, but their underlying patterns heavily influence the time of day people rent bikes.

### Modeling Implication
Like weekday, `workingday` is likely a strong predictor when combined with hour. It allows the model to differentiate between commute-heavy days and leisure-heavy days.

## 13. Month Analysis
### Question
Does demand change gradually throughout the year?

### Code

In [ ]:
monthly_stats = df.groupby("mnth")["cnt"].agg(["mean", "median", "count"]).reset_index()
print("Demand by Month:")
display(monthly_stats)

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_stats, x="mnth", y="mean", color="teal")
plt.title("Average Bike Demand by Month")
plt.xlabel("Month (1-12)")
plt.ylabel("Average Rentals")
plt.show()


### Observation
Demand typically follows a bell curve over the year: low in Jan/Feb, peaking around Jun-Sep, and dropping back down towards Dec.

### Modeling Implication
Month contains granular seasonal information beyond just the 4 macro-seasons. It is a useful predictor, and cyclical encoding for month can also be beneficial.

## 14. Year Analysis
### Question
Is there a long-term trend in bike usage across different years?

### Code

In [ ]:
year_stats = df.groupby("yr")["cnt"].agg(["mean", "median", "count"]).reset_index()
print("Demand by Year:")
display(year_stats)

plt.figure(figsize=(6, 5))
sns.barplot(data=year_stats, x="yr", y="mean")
plt.title("Average Rental Demand by Year")
plt.xlabel("Year Indicator (0, 1, ...)")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
If bike usage increased significantly from year 0 to year 1, this indicates long-term growth or increasing popularity of the service.

### Modeling Implication
A strong year effect is a very important trend predictor. However, if predicting future years (e.g., year 2 or 3), the model might struggle to extrapolate this trend unless year is treated carefully.

## 15. Temperature vs Rentals
### Question
How does temperature relate to rental demand?

### Code

In [ ]:
plt.figure(figsize=(8, 5))
# Sample data for scatter plot if dataset is very large
sample_df = df.sample(frac=0.1, random_state=42) if len(df) > 10000 else df
sns.regplot(data=sample_df, x="temp", y="cnt", scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("Temperature vs Rental Demand")
plt.xlabel("Normalized Temperature (temp)")
plt.ylabel("Rentals (cnt)")
plt.show()

corr_temp = df['temp'].corr(df['cnt'])
print(f"Correlation between temp and cnt: {corr_temp:.3f}")


### Observation
There is generally a positive correlation between temperature and demand: higher temperatures correspond to higher demand, up to a certain point (extreme heat might cause a drop, though linear correlation won't capture non-linearity perfectly).

### Modeling Implication
Temperature appears to be a strong numerical predictor. We might want to check for non-linear patterns (e.g., polynomial features) if extreme heat reduces demand.

## 16. Feels-Like Temperature
### Question
Does feels-like temperature provide different information than actual temperature?

### Code

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=sample_df, x="atemp", y="cnt", scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("Feels-Like Temperature vs Rental Demand")
plt.xlabel("Feels-Like Temperature (atemp)")
plt.ylabel("Rentals (cnt)")
plt.show()

corr_atemp = df['atemp'].corr(df['cnt'])
corr_temp_atemp = df['temp'].corr(df['atemp'])
print(f"Correlation between atemp and cnt: {corr_atemp:.3f}")
print(f"Correlation between temp and atemp: {corr_temp_atemp:.3f}")


### Observation
`atemp` and `temp` have extremely similar relationships with `cnt`. More importantly, `temp` and `atemp` are usually extremely highly correlated (approaching 1.0).

### Modeling Implication
**Multicollinearity:** Having both `temp` and `atemp` in the model introduces redundancy and can destabilize linear model coefficients. We may consider dropping one (usually `atemp`) before modeling.

## 17. Humidity vs Rentals
### Question
How does humidity affect bike rentals?

### Code

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=sample_df, x="hum", y="cnt", scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("Humidity vs Rental Demand")
plt.xlabel("Normalized Humidity (hum)")
plt.ylabel("Rentals (cnt)")
plt.show()

corr_hum = df['hum'].corr(df['cnt'])
print(f"Correlation between hum and cnt: {corr_hum:.3f}")


### Observation
Humidity typically has a negative correlation with demand; higher humidity (especially above certain thresholds) makes biking less comfortable, lowering rentals.

### Modeling Implication
Humidity acts as a moderate negative predictor of demand.

## 18. Windspeed vs Rentals
### Question
How does windspeed relate to rental demand?

### Code

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=sample_df, x="windspeed", y="cnt", scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("Windspeed vs Rental Demand")
plt.xlabel("Normalized Windspeed (windspeed)")
plt.ylabel("Rentals (cnt)")
plt.show()

corr_wind = df['windspeed'].corr(df['cnt'])
print(f"Correlation between windspeed and cnt: {corr_wind:.3f}")


### Observation
Windspeed often has a weak negative correlation with rentals. The distribution of windspeed may appear discrete (taking specific repeated values), which could indicate the precision of the anemometer used.

### Modeling Implication
Windspeed is a weak to moderate predictor. The discrete nature might not be a major issue, but we should be aware of it.

## 19. Correlation Analysis
### Question
What are the overall linear relationships between numerical features and the target?

### Code

In [ ]:
numerical_cols = ['cnt', 'temp', 'atemp', 'hum', 'windspeed', 'hr', 'season', 'mnth', 'yr', 'workingday', 'holiday', 'casual', 'registered']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()


### Observation
- `casual` and `registered` perfectly correlate with `cnt` combined (Leakage).
- `temp` and `atemp` are highly correlated (Multicollinearity).
- `hr`, `temp`, `yr` usually show stronger correlations with `cnt`.
- `hum` and `weathersit` (if numerical) show negative correlations.

### Modeling Implication
We must drop `casual` and `registered`. We should likely drop `atemp` to fix multicollinearity. Strong numerical correlations point to feature importance, but linear correlation misses cyclical patterns (like `hr`).

## 20. Numerical Feature Distributions
### Question
How are the numerical features distributed? Are there skewed variables or outliers?

### Code

In [ ]:
num_features = ['temp', 'atemp', 'hum', 'windspeed', 'cnt']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')

# Remove empty subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()


### Observation
- `temp` and `atemp` are usually normally distributed.
- `hum` might have some zero values or left skew.
- `windspeed` may be right-skewed and discrete.
- `cnt` is heavily right-skewed.

### Modeling Implication
The skewness in `cnt` and `windspeed` might warrant transformations (like log) to help linear models. Identifying zero values in humidity can help verify data quality.

## 21. Categorical Feature Distributions
### Question
Are categorical features well-balanced?

### Code

In [ ]:
cat_features = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    sns.countplot(data=df, x=col, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Count Plot of {col}')

fig.delaxes(axes[8])
plt.tight_layout()
plt.show()


### Observation
Most time-based categories (month, hour, weekday) are perfectly balanced. `holiday` is heavily imbalanced (mostly 0s), and `weathersit` category 4 (Severe Weather) contains extremely few observations.

### Modeling Implication
Rare categories (holiday, severe weather) can be tricky for cross-validation splits as they might missing from a fold. These imbalances reflect reality but require robust models.

## 22. Outlier Analysis
### Question
Do numerical features or the target contain significant outliers?

### Code

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.boxplot(data=df, x=col, ax=axes[i], color='lightgreen')
    axes[i].set_title(f'Boxplot of {col}')

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()


### Observation
`cnt`, `windspeed`, and potentially `hum` (low values) show many statistical "outliers" beyond the IQR whiskers.

### Modeling Implication
In this context, high `cnt` values (spikes in demand) are likely valid extreme observations representing rush hours or perfect weather days, not data errors. Therefore, we should **NOT** automatically remove these outliers. They are the actual demand peaks we want to predict.

## 23. Interaction Analysis
### Question
Do important features interact with each other (e.g., Hour and Working Day)?

### Code

In [ ]:
# Hour x Working Day
plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='hr', y='cnt', hue='workday_name', estimator='mean', ci=None)
plt.title("Hourly Demand Pattern: Working Day vs Non-working Day")
plt.xlabel("Hour of the Day")
plt.ylabel("Mean Rentals")
plt.show()

# Hour x Season
plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='hr', y='cnt', hue='season_name', estimator='mean', ci=None)
plt.title("Hourly Demand Pattern by Season")
plt.xlabel("Hour of the Day")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
- **Hour × Working Day**: Commuter peaks (8 AM, 5 PM) only exist on working days. Non-working days have a single broad peak in the middle of the afternoon.
- **Hour × Season**: The shape of the daily pattern remains somewhat similar across seasons, but the overall volume is scaled up or down based on the season.

### Modeling Implication
Tree-based models can capture these interactions naturally. If using linear models, we would need to explicitly engineer interaction terms (e.g., `is_workingday * hour`) to capture these drastically different hourly profiles.

## 24. Registered vs Casual Users
### Question
How do casual and registered user demands compare?

### Code

In [ ]:
casual_mean = df['casual'].mean()
registered_mean = df['registered'].mean()
total_mean = df['cnt'].mean()

print(f"Mean Casual Rentals: {casual_mean:.2f} ({casual_mean/total_mean*100:.1f}%)")
print(f"Mean Registered Rentals: {registered_mean:.2f} ({registered_mean/total_mean*100:.1f}%)")

plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='hr', y='casual', label='Casual', ci=None)
sns.lineplot(data=df, x='hr', y='registered', label='Registered', ci=None)
plt.title("Casual vs Registered Hourly Pattern")
plt.xlabel("Hour of the Day")
plt.ylabel("Mean Rentals")
plt.show()


### Observation
Registered users dominate the total demand (~80%) and drive the commuter peaks. Casual users have a smoother afternoon peak, similar to weekend patterns.

### Modeling Implication
**DO NOT use `casual` and `registered` as predictors for `cnt`.** This is purely for business understanding. However, the different profiles suggest that building two separate models (one for casual, one for registered) and summing their predictions might yield better results than predicting `cnt` directly.

## 25. Time-Based Feature Engineering Opportunities
### Question
What features can we extract or engineer from the temporal structure?

### Code

In [ ]:
# Example of cyclical encoding logic
print("Potential engineered features:")
print("- hour_sin = np.sin(2 * np.pi * hr / 24)")
print("- hour_cos = np.cos(2 * np.pi * hr / 24)")
print("- month_sin = np.sin(2 * np.pi * mnth / 12)")
print("- is_weekend = 1 if weekday in [0, 6] else 0")


### Observation
Time features like hour and month are cyclical. 23:00 is mathematically far from 00:00, but chronologically adjacent.

### Modeling Implication
Cyclical encoding maps these periods onto a circle using sine and cosine, ensuring regression models understand the periodic relationship. We can also engineer simpler binary features like `is_weekend` or `is_rush_hour` based on domain knowledge.

## Potential Modeling Risks

### Target Leakage
`casual` and `registered` directly construct `cnt` (`cnt = casual + registered`). They must be excluded from feature sets when predicting `cnt`.

### Multicollinearity
`temp` and `atemp` are highly correlated. Including both might destabilize linear models; one should typically be dropped.

### Temporal Dependence
The observations are ordered chronologically. A random train-test split will leak future patterns into training and artificially inflate validation scores. A chronological split (e.g., predicting the last few months based on previous months) is strongly recommended.

### Seasonality
The data has daily, weekly, and yearly seasonality. Models need adequate time features to capture this.

### Target Skewness
The target `cnt` is heavily right-skewed. Transforming it (e.g., `np.log1p(cnt)`) could stabilize variance for regression algorithms.

### Imbalanced Categories
Holidays and extreme weather (category 4) are rare. Stratified sampling (if not using time-based split) or robust evaluation metrics are needed.

### Outliers
High rental counts are legitimate demand peaks (valid extreme observations), not data errors. They should not be removed.

# EDA Conclusions

## Target Behavior
The target `cnt` is right-skewed with significant temporal patterns. It shows distinct daily seasonality (commuter peaks) and yearly seasonality (summer vs winter). 

## Strong Predictors
- **Hour (`hr`)**: Defines the commuter peaks.
- **Temperature (`temp`)**: Strongly correlates with biking comfort and demand.
- **Season (`season`) / Month (`mnth`)**: Captures macro weather patterns.
- **Year (`yr`)**: Captures long-term growth trends.
- **Working Day (`workingday`)**: Modifies the hourly pattern heavily.

## Moderate Predictors
- **Weather Situation (`weathersit`)**: Drops demand in bad weather.
- **Humidity (`hum`)**: Higher humidity generally lowers demand.
- **Weekday (`weekday`)**: Differentiates weekends from weekdays.

## Weak Predictors
- **Windspeed (`windspeed`)**: Slight negative impact, relatively weak compared to temperature.

## Data Quality
- The dataset generally requires verifying missing values, deduplicating potential identical timestamps, and ensuring categorical constraints (e.g. valid month ranges) are met. (Outputs will specify if fixes are required). Outliers in `cnt` are legitimate.

## Leakage Risks
- **`casual` and `registered` must be dropped.**

## Feature Engineering Opportunities
- Cyclical encoding for `hr` and `mnth`.
- Weekend / Rush Hour indicators.
- Interactions (Hour × Workingday) for linear models.

## Modeling Recommendations
1. Remove target-leakage variables (`casual`, `registered`).
2. Separate features and target.
3. Check for multicollinearity (drop `atemp`).
4. Create time-based train/validation/test splits (do not shuffle).
5. Build a preprocessing pipeline (impute if necessary, scale numerical features, encode categoricals).
6. Engineer cyclical time features.
7. Build a simple baseline regression model (e.g., Linear Regression, Ridge).
8. Compare models (e.g., adding a Random Forest or XGBoost to handle non-linearities and interactions naturally) using appropriate regression metrics (RMSE, MAE, R²).
9. Perform cross-validation appropriate for temporal data (e.g., TimeSeriesSplit).
10. Tune hyperparameters only after establishing a solid baseline.

### Summary Table

| Feature | Type | Relationship with Target | Predictive Potential | Issues | Recommended Action |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **hr** | Categorical/Cyclical | Strong commuter peaks | Strong | Cyclical boundary | Cyclical encoding (sin/cos) |
| **temp** | Numerical | Positive correlation | Strong | - | Scale/Keep |
| **yr** | Categorical | Positive long-term trend | Strong | Generalization | Keep, handle carefully |
| **workingday**| Binary | Interacts strongly with hour | Strong | - | Keep, maybe engineer interactions |
| **season/mnth**| Categorical/Cyclical | Bell curve peaking in summer | Strong | Cyclical boundary | Keep / Cyclical encode month |
| **atemp** | Numerical | Positive correlation | Strong | Multicollinearity | Drop |
| **weathersit**| Categorical | Negative correlation for bad weather | Moderate | Imbalanced categories | Keep, monitor rare classes |
| **hum** | Numerical | Negative correlation | Moderate | - | Scale/Keep |
| **weekday** | Categorical | Defines weekend vs weekday | Moderate | - | Keep / One-hot encode |
| **windspeed** | Numerical | Weak negative correlation | Weak | - | Scale/Keep |
| **holiday** | Binary | Varies | Weak | Highly imbalanced | Keep, but rely more on workingday |
| **casual** | Numerical | Target component | Leakage / Do Not Use| Target Leakage | Drop |
| **registered**| Numerical | Target component | Leakage / Do Not Use| Target Leakage | Drop |

### Final Data Science Assessment
1. **What drives bike rental demand?** Daily commuting schedules, seasonal weather (temperature), and long-term service adoption (year).
2. **Which features appear most useful?** Hour, Temperature, Year, Working Day, Season/Month.
3. **Which features are weak?** Windspeed, Holiday.
4. **Which features create leakage?** Casual, Registered.
5. **What preprocessing is needed?** Dropping leakage variables, handling multicollinearity (drop atemp), scaling numericals, encoding categoricals.
6. **What feature engineering should be considered?** Cyclical encoding for time variables, explicit interaction terms for linear models.
7. **What validation strategy should be used?** Time-based chronological split (e.g., TimeSeriesSplit) to respect the temporal dependence and avoid data leakage from the future.
8. **What should be the next step before model training?** Implement the preprocessing pipeline and create the chronological train-test split.